In [57]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator #Genera datos (imagenes) sintéticos
from keras import optimizers #Técnica del descenso del gradiente, sirve para minimizar el error en el proceso de entrenamiento
from keras.models import Sequential # Estacebe un modelo de red neuronal por capas
from keras.layers import Dense, Flatten, Dropout, Activation
#Dense = define las neuronas por cada capa
#Flatter = aplana los datos en formato matriz N-Tensor --> 1-Tensor
#Dropout = técnica de reducción del sobreajuste o sobre entrenamiento (apagar % de neuronas)
#Activation = determina las funciones de activiación en cada neurona (Relu, sigmoid, softmax, tanh, linear, etc)
from keras.layers import Convolution2D, MaxPooling2D

In [46]:
# APARTIR DE AQUÍ SE MANEJARA LO NECESARIO PARA LA IMPLEMENTACIÓN DE LA INTERFAZ GRAFICA.
#PILLOW: PROCESAMIENTO DE IMAGENES 

import tkinter as tk  # IMPORTAMOS TKINTER PARA CREAR INTERFACES GRÁFICAS (VENTANAS, BOTONES, ETC.)
from tkinter import ttk, filedialog, messagebox, scrolledtext  # IMPORTAMOS COMPONENTES AVANZADOS DE TKINTER: ESTILOS, DIÁLOGOS Y ÁREAS DE TEXTO DESPLAZABLES

import threading  # IMPORTAMOS THREADING PARA EJECUTAR PROCESOS EN SEGUNDO PLANO SIN CONGELAR LA INTERFAZ
import queue  # IMPORTAMOS QUEUE PARA MANEJAR COLAS DE MENSAJES ENTRE HILOS (THREADS)
import os  # IMPORTAMOS OS PARA INTERACTUAR CON EL SISTEMA OPERATIVO (ARCHIVOS, RUTAS, ETC.)
import time  # IMPORTAMOS TIME PARA MANEJAR TIEMPOS, PAUSAS Y MEDICIONES

from concurrent.futures import ThreadPoolExecutor  # IMPORTAMOS THREADPOOL PARA EJECUTAR TAREAS CON HILOS DE FORMA EFICIENTE
from datetime import datetime  # IMPORTAMOS DATETIME PARA MANEJO DE FECHAS Y HORAS

from dataclasses import dataclass  # IMPORTAMOS DATACLASS PARA CREAR CLASES SIMPLES PARA ALMACENAR DATOS
from enum import Enum, auto  # IMPORTAMOS ENUM PARA CREAR ENUMERACIONES (CONSTANTES AGRUPADAS)
import cv2
from PIL import Image, ImageTk  # IMPORTAMOS CV2 PARA PROCESAMIENTO DE VIDEO Y CÁMARA, Y PILLOW PARA MANEJO DE IMÁGENES EN LA INTERFAZ

from typing import Optional, Callable  # IMPORTAMOS TYPING PARA DEFINIR TIPOS OPCIONALES Y FUNCIONES COMO PARÁMETROS

import sys  # IMPORTAMOS SYS PARA ACCEDER A FUNCIONES DEL SISTEMA Y PARÁMETROS DEL INTÉRPRETE DE PYTHON

In [ ]:
# /*APARTIR DE AQUÍ SE CONSTRUIRÁ todo LO RELACIONADO A LA INTERFAZ - LA INTERFAZ SE ENCUENTRA EL EL ARCHIVO .py*/

APARTIR DE ACA ABAJO DE CONSTRUIRÁ LA RED NEURONAL

In [47]:
#Definir los hiperparámetros de la red neurona convolucional --
#Definir la ruta de los datos de entranamiento
entrenar = "CNN_Imagenes/entrenar"
validar = "CNN_Imagenes/validar"
#Hiperparámetros
#epocas = 50  # MEJORA (NO MUCHO)
epocas = 100 # Mejora el entrenamiento al aumentar las épocas - 2026/04/17
#altura,anchura = 200,200 #ORIGINAL - 20260414 
altura, anchura = 600, 600 # MEJORA EL ENTRENAMIENTO AL AUMENTAR LA RESOLUCIÓN DE LAS IMÁGENES - 2026/04/17
batch_size = 8 #  Mejora el entrenamiento 
pasos = 564 // batch_size # Evita errores de entrenamiento - Dividirlo sobre el batch_size para que el número de pasos sea un entero y permita un entrenamiento fluido

#Definir la cantidad de kernels por cada capa
#Nuestro primer kernel
kernel1=32 # 2,4,8,16,32,64,128,256, 512, etc (Estos son como se )
kernel1_size = (3,3)
#Nuestro segundo kernel
kernel2=64
kernel2_size = (3,3)
#Nuestro tercer kernel
kernel3 = 128
kernel3_size = (3,3)
#KERNEL 4
kernel4 = 128
kernel4_size = (3,3)
size_pooling = (2,2) #Volvemos la matríz mas pequeña para mejorar el análisis
# clases = 6 #Número de objetos a detectar o identificar - SON 6 LAS QUE SE MANEJAN EN LAS CARPETAS
clases = 8 #Mejora el entrenamiento al agregar una clase más (LA DE BASURA DEBAJO DEL AGUA) e incluimos una clase para imágenes no relacionadas con contaminación hídrica (NO_RELACIONADO) - 20260414

In [49]:
#Generar datos sintéticos (si la cantidad de datos es pequeña) 
#Esto permite aumentar la cantidad de datos de entrenamiento a partir de las imágenes originales, aplicando transformaciones como rotación, zoom, volteo, etc. 
# Ayudando a mejorar la generalización del modelo y reducir el sobreajuste.
entrenamiento = ImageDataGenerator(
                            rescale=1/255, # Normaliza los valores de píxeles a un rango de 0 a 1 para mejorar la convergencia del modelo
                            zoom_range=0.2, # Aplica zoom aleatorio hasta un 20% para simular diferentes distancias de cámara y mejorar la robustez del modelo
                            horizontal_flip=True, # Voltea horizontalmente las imágenes para simular diferentes orientaciones y aumentar la diversidad del conjunto de entrenamiento
                            rotation_range=15,  # Se asigna 15° para simular ligeras inclinaciones sin distorsionar la realidad
                            width_shift_range=0.1,  # Desplaza la imagen horizontalmente un 10% para simular movimiento de cámara
                            height_shift_range=0.1,  # Desplaza verticalmente un 10% para robustecer el modelo ante encuadres distintos
                            shear_range=0.1,  # Aplica deformaciones leves (tipo inclinación) para mejorar generalización
                            brightness_range=[0.8,1.2]  # Ajusta brillo entre 80% y 120% para simular condiciones de iluminación reales
                            )
# NOTA.- Se eligieron transformaciones leves para evitar alterar la naturaleza real del agua,
# pero suficientes para mejorar la generalización del modelo ante variaciones de iluminación, ángulo y posición.”

#Posteriormente se aplicará solo la normalización a las imágenes de validación, ya que no queremos introducir variaciones artificiales
#  en el conjunto de validación que debe reflejar datos reales para evaluar correctamente el rendimiento del modelo.
validacion = ImageDataGenerator(rescale=1/255) #Normaliza las imagenes de validación
#Extraer las imagenes de las carpetas
#Obtenemos las imágenes de entrenamiento y validación a partir de las carpetas, aplicando las transformaciones definidas en los generadores de datos.
imagenes_entrenamiento = entrenamiento.flow_from_directory(entrenar,
                                                    target_size=(anchura,altura),
                                                    batch_size=batch_size,
                                                    class_mode="categorical") #Aplicamos categorical porque se cuenta con 8 clases - En entrenamiento 
imagenes_validacion = validacion.flow_from_directory(validar,
                                                target_size=(anchura,altura),
                                                batch_size=batch_size,
                                                class_mode="categorical") #Aplicamos categorical porque se cuenta con 8 clases - En validación

Found 3601 images belonging to 8 classes.
Found 1044 images belonging to 8 classes.


In [50]:
#Definir la arquitectura de la red neuronal convolucional - AQUÍ SE ENCONTRARÁN LAS CAPAS OCULTAS --

# SE USARÁ LA FUNCIÓN DE ACTIVACIÓN RELU (Rectified Linear Unit o Unidad Lineal Rectificada) 
# porque devuelve 0 para entradas negativas y el mismo valor para entradas positivas.

#Tambien se usará la función de activación softmax en la capa de salida, ya que es adecuada para problemas de clasificación multiclase,
# porque convierte las salidas de la red en probabilidades que suman 1, lo que facilita la interpretación de las predicciones como probabilidades de pertenencia a cada clase.

CNN = Sequential() #Creamos nuestro modelo de red neuronal convolucional por capas (Sequential) para ir agregando las 
                    #capas de la red neuronal convolucional de forma secuencial (una después de otra)
#Capa 1 - La primera capa convolucional se encarga de extraer características básicas de las imágenes, 
# como bordes y texturas, utilizando 32 filtros de tamaño 3x3 con activación ReLU. 
# Luego, la capa de submuestreo (MaxPooling) para reducir la dimensión de las imágenes y destacar (maximizar) lo mas importante
CNN.add(Convolution2D(kernel1,
                    kernel1_size,
                    padding="same",
                    input_shape=(altura,anchura,3),
                    activation="relu")) #Primera capa convolucional
CNN.add(MaxPooling2D(pool_size=size_pooling)) #Capa de submuestreo 

#Capa 2
CNN.add(Convolution2D(kernel2,
                    kernel2_size,
                    padding="same",
                    activation="relu")) #Segunda capa convolucional
CNN.add(MaxPooling2D(pool_size=size_pooling)) #Capa de submuestreo 

#Capa 3
CNN.add(Convolution2D(kernel3,
                    kernel3_size,
                    padding="same",
                    activation="relu")) #Tercera capa convolucional
CNN.add(MaxPooling2D(pool_size=size_pooling)) #Capa de submuestreo

#Capa 4
CNN.add(Convolution2D(kernel4,
                    kernel4_size,
                    padding="same",
                    activation="relu")) #Cuarta capa convolucional
CNN.add(MaxPooling2D(pool_size=size_pooling)) #Capa de submuestreo

#Aplanar las matrices en formato de vector
CNN.add(Flatten())

#Conectar a el perceptrón multicapa
CNN.add(Dense(128,activation="relu")) #Agregamos una capa densa con 128 neuronas y función de activación ReLU para aprender patrones complejos a partir de las características extraídas por las capas convolucionales.
CNN.add(Dense(64,activation="relu")) #Agregamos una segunda capa densa con 64 neuronas y función de activación ReLU para seguir aprendiendo patrones más complejos y reducir la dimensionalidad antes de la capa de salida.

CNN.add(Dropout(0.5)) #Agregamos una capa de Dropout de 5% (0.5) para reducir el overlifting 
# o sobre entrenamiento, al apagar aleatoriamente neuronas durante el entrenamiento.

#Definir la capa de salida
CNN.add(Dense(clases,activation="softmax")) #Agregamos la capa de salida con un número de neuronas igual al número de clases
#(8) y función de activación softmax para obtener probabilidades de clasificación multiclase.


c:\Users\alexi\OneDrive\Desktop\SII_CNN\CNN_Agua\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [58]:
#Definir los parámetros del entrenamiento
CNN.compile(loss="categorical_crossentropy",optimizer="adam",metrics=["accuracy","mse"])
#La función de pérdida "categorical_crossentropy" permite la medición de diferencia entre las distribuciones de probabilidad predicha y real.

#Ademas, hacemos uso de la función de optimización "adam" que es un algoritmo de optimización eficiente y ampliamente utilizado en el entrenamiento de redes neuronales,
# que combina las ventajas de los métodos de descenso de gradiente adaptativo y el momento para
#Metricas usadas accuracy que permite medir la precisión del modelo y mse (mean squared error) que mide el error cuadrático medio entre las predicciones y las etiquetas reales.

In [ ]:
# #Realizamos el entrenamiento ----------
# /*NOTA. LA CLASE AGUA_CON_PARTICULAS (TURBIA / LODO / SEDIMENTO) ES ESO; ademas de clase NO_RELACIONADO (todo lo que NO tiene que ver con agua)*/
import numpy as np #Para hacer uso de funciones matemáticas
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping #Agregamos callbacks para guardar el mejor modelo durante el entrenamiento 
#y para detener el entrenamiento si no hay mejora en la validación después de cierto número de épocas (EarlyStopping)

#Guarda automaticamente el mejor modelo durante el entrenamiento (Usado por sí no se incrementaba la precisión y
#se quedaba toda la noche)
# Este checkpoint guarda automáticamente el MEJOR MODELO COMPLETO (.h5)
checkpoint = ModelCheckpoint(
    "CNN_Imagenes/Modelo/cnn_mejor.h5",  # Ruta para guardar el mejor modelo
    monitor='val_loss',  # Monitorea el error de validación (error en validación)
    save_best_only=True,  # Guarda solo el mejor modelo 
    mode='min',  # Minimiza el error de validación
    verbose=1 # Muestra mensajes en consola
)

# Este checkpoint guarda SOLO LOS PESOS del mejor modelo
checkpoint_weights = ModelCheckpoint(
    "CNN_Imagenes/Modelo/cnn_pesos_mejor.weights.h5", # Muestra mensajes en consola
    monitor='val_loss',
    save_best_only=True,
    mode='min',
    save_weights_only=True,  # Guarda solo los pesos - A diferencia del anterior que guarda todo el modelo
    verbose=1
)
#En caso de que NO MEJORE (Dejará de analizar)
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,  # Espera 10 épocas sin mejora
    restore_best_weights=True
)
#--------------------------------------------------------
#Obtenemos las clases y el conteo de imágenes por clase para calcular los pesos de clase, 
# lo que ayuda a manejar el desequilibrio en el conjunto de datos durante el entrenamiento.
# -
# Obtener etiquetas de cada imagen en el conjunto de entrenamiento
clases_array = imagenes_entrenamiento.classes
# Relizamos el conteo de la cantidad de imágenes por clase utilizando np.bincount (encargado de contar).
conteo = np.bincount(clases_array)
# Calculamos el total de imágenes
total = np.sum(conteo)
# Número de clases
num_clases = len(conteo)
# Realizamos un arreglo de pesos (el cual será encargado de almacenar los pesos con los que cuenta CADA clase)
class_weights = {}

# Calculamos el peso de cada clase
# Hacemos uso del total / (cantClases * imagenesDeCadaClase)
# Esto hace que las clases con menos imágenes tengan más peso y el aprendizaje se enfoque en ellas (más)
for i in range(num_clases):
    class_weights[i] = total / (num_clases * conteo[i])

#Presentamos en terminal cuanto peso tiene cada clase (del cálculo previo)
print("Pesos por clase:", class_weights)

#Finalizando con el entrenamiento
historico = CNN.fit(imagenes_entrenamiento, # Datos de entrenamiento
                    validation_data=imagenes_validacion, #Datos o imágenes de validación
                    epochs=epocas, #Cant. de epocas
                    validation_steps=imagenes_validacion.samples // batch_size, # Pasos de validación
                    verbose=1, #Presenta en pantalla el progreso
                    class_weight=class_weights, # Aplica pesos a las clases
                    callbacks=[checkpoint, checkpoint_weights])  # Guarda mejor modelo y pesos (en caso de encontrar)

Pesos por clase: {0: np.float64(1.0846385542168675), 1: np.float64(1.1225062344139651), 2: np.float64(1.0253416856492028), 3: np.float64(0.8931051587301587), 4: np.float64(0.9958517699115044), 5: np.float64(0.8966633466135459), 6: np.float64(0.9223872950819673), 7: np.float64(1.1253125)}
Epoch 1/100
451/451 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.2076 - loss: 2.0374 - mse: 0.1061
Epoch 1: val_loss improved from None to 1.56903, saving model to CNN_Imagenes/Modelo/cnn_mejor.h5



Epoch 1: finished saving model to CNN_Imagenes/Modelo/cnn_mejor.h5

Epoch 1: val_loss improved from None to 1.56903, saving model to CNN_Imagenes/Modelo/cnn_pesos_mejor.weights.h5

Epoch 1: finished saving model to CNN_Imagenes/Modelo/cnn_pesos_mejor.weights.h5
451/451 ━━━━━━━━━━━━━━━━━━━━ 1076s 2s/step - accuracy: 0.2666 - loss: 1.9022 - mse: 0.1006 - val_accuracy: 0.3933 - val_loss: 1.5690 - val_mse: 0.0881
Epoch 2/100
451/451 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.3406 - loss: 1.6583 - mse: 0.0923
Epoch 2: val_loss did not improve from 1.56903

Epoch 2: val_loss did not improve from 1.56903
451/451 ━━━━━━━━━━━━━━━━━━━━ 877s 2s/step - accuracy: 0.3546 - loss: 1.6359 - mse: 0.0913 - val_accuracy: 0.3990 - val_loss: 1.6817 - val_mse: 0.0925
Epoch 3/100
451/451 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.4098 - loss: 1.5947 - mse: 0.0882
Epoch 3: val_loss improved from 1.56903 to 1.52752, saving model to CNN_Imagenes/Modelo/cnn_mejor.h5



Epoch 3: finished saving model to CNN_Imagenes/Modelo/cnn_mejor.h5

Epoch 3: val_loss improved from 1.56903 to 1.52752, saving model to CNN_Imagenes/Modelo/cnn_pesos_mejor.weights.h5

Epoch 3: finished saving model to CNN_Imagenes/Modelo/cnn_pesos_mejor.weights.h5
451/451 ━━━━━━━━━━━━━━━━━━━━ 830s 2s/step - accuracy: 0.4146 - loss: 1.6002 - mse: 0.0877 - val_accuracy: 0.4346 - val_loss: 1.5275 - val_mse: 0.0849
Epoch 4/100
451/451 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.4811 - loss: 1.4006 - mse: 0.0803
Epoch 4: val_loss improved from 1.52752 to 1.24483, saving model to CNN_Imagenes/Modelo/cnn_mejor.h5



Epoch 4: finished saving model to CNN_Imagenes/Modelo/cnn_mejor.h5

Epoch 4: val_loss improved from 1.52752 to 1.24483, saving model to CNN_Imagenes/Modelo/cnn_pesos_mejor.weights.h5

Epoch 4: finished saving model to CNN_Imagenes/Modelo/cnn_pesos_mejor.weights.h5
451/451 ━━━━━━━━━━━━━━━━━━━━ 835s 2s/step - accuracy: 0.4810 - loss: 1.3979 - mse: 0.0799 - val_accuracy: 0.5596 - val_loss: 1.2448 - val_mse: 0.0738
Epoch 5/100
451/451 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.5008 - loss: 1.3404 - mse: 0.0777
Epoch 5: val_loss improved from 1.24483 to 1.22963, saving model to CNN_Imagenes/Modelo/cnn_mejor.h5



Epoch 5: finished saving model to CNN_Imagenes/Modelo/cnn_mejor.h5

Epoch 5: val_loss improved from 1.24483 to 1.22963, saving model to CNN_Imagenes/Modelo/cnn_pesos_mejor.weights.h5

Epoch 5: finished saving model to CNN_Imagenes/Modelo/cnn_pesos_mejor.weights.h5
451/451 ━━━━━━━━━━━━━━━━━━━━ 822s 2s/step - accuracy: 0.4932 - loss: 1.3546 - mse: 0.0781 - val_accuracy: 0.5712 - val_loss: 1.2296 - val_mse: 0.0727
Epoch 6/100
451/451 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.5408 - loss: 1.2516 - mse: 0.0733
Epoch 6: val_loss did not improve from 1.22963

Epoch 6: val_loss did not improve from 1.22963
451/451 ━━━━━━━━━━━━━━━━━━━━ 804s 2s/step - accuracy: 0.5410 - loss: 1.2538 - mse: 0.0730 - val_accuracy: 0.5817 - val_loss: 1.2298 - val_mse: 0.0720
Epoch 7/100
451/451 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.5377 - loss: 1.2317 - mse: 0.0722
Epoch 7: val_loss improved from 1.22963 to 1.19994, saving model to CNN_Imagenes/Modelo/cnn_mejor.h5


...


Epoch 27: finished saving model to CNN_Imagenes/Modelo/cnn_mejor.h5

Epoch 27: val_loss improved from 0.87163 to 0.86577, saving model to CNN_Imagenes/Modelo/cnn_pesos_mejor.weights.h5

Epoch 27: finished saving model to CNN_Imagenes/Modelo/cnn_pesos_mejor.weights.h5
451/451 ━━━━━━━━━━━━━━━━━━━━ 800s 2s/step - accuracy: 0.6906 - loss: 0.9001 - mse: 0.0529 - val_accuracy: 0.7471 - val_loss: 0.8658 - val_mse: 0.0454
Epoch 28/100
451/451 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.7069 - loss: 0.8342 - mse: 0.0512
Epoch 28: val_loss did not improve from 0.86577

Epoch 28: val_loss did not improve from 0.86577
451/451 ━━━━━━━━━━━━━━━━━━━━ 795s 2s/step - accuracy: 0.6951 - loss: 0.8782 - mse: 0.0522 - val_accuracy: 0.7154 - val_loss: 0.9984 - val_mse: 0.0531
Epoch 29/100
451/451 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.6918 - loss: 0.8665 - mse: 0.0519
Epoch 29: val_loss did not improve from 0.86577

Epoch 29: val_loss did not improve from 0.86577
451/451 ━━━━━━━━━━━━━━━━━━━━ 789s 2


Epoch 45: finished saving model to CNN_Imagenes/Modelo/cnn_mejor.h5

Epoch 45: val_loss improved from 0.86577 to 0.80949, saving model to CNN_Imagenes/Modelo/cnn_pesos_mejor.weights.h5

Epoch 45: finished saving model to CNN_Imagenes/Modelo/cnn_pesos_mejor.weights.h5
451/451 ━━━━━━━━━━━━━━━━━━━━ 793s 2s/step - accuracy: 0.7081 - loss: 0.8622 - mse: 0.0497 - val_accuracy: 0.7692 - val_loss: 0.8095 - val_mse: 0.0417
Epoch 46/100
451/451 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.7444 - loss: 0.7355 - mse: 0.0446
Epoch 46: val_loss did not improve from 0.80949

Epoch 46: val_loss did not improve from 0.80949
451/451 ━━━━━━━━━━━━━━━━━━━━ 790s 2s/step - accuracy: 0.7373 - loss: 0.7567 - mse: 0.0452 - val_accuracy: 0.7260 - val_loss: 1.0590 - val_mse: 0.0501
Epoch 47/100
451/451 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.7204 - loss: 0.8207 - mse: 0.0476
Epoch 47: val_loss did not improve from 0.80949

Epoch 47: val_loss did not improve from 0.80949
451/451 ━━━━━━━━━━━━━━━━━━━━ 789s 2


Epoch 49: finished saving model to CNN_Imagenes/Modelo/cnn_mejor.h5

Epoch 49: val_loss improved from 0.80949 to 0.76192, saving model to CNN_Imagenes/Modelo/cnn_pesos_mejor.weights.h5

Epoch 49: finished saving model to CNN_Imagenes/Modelo/cnn_pesos_mejor.weights.h5
451/451 ━━━━━━━━━━━━━━━━━━━━ 796s 2s/step - accuracy: 0.7317 - loss: 0.7890 - mse: 0.0472 - val_accuracy: 0.7596 - val_loss: 0.7619 - val_mse: 0.0413
Epoch 50/100
451/451 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.7434 - loss: 0.7353 - mse: 0.0447
Epoch 50: val_loss did not improve from 0.76192

Epoch 50: val_loss did not improve from 0.76192
451/451 ━━━━━━━━━━━━━━━━━━━━ 789s 2s/step - accuracy: 0.7515 - loss: 0.7187 - mse: 0.0433 - val_accuracy: 0.7423 - val_loss: 0.9462 - val_mse: 0.0445
Epoch 51/100
451/451 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.7530 - loss: 0.7337 - mse: 0.0432
Epoch 51: val_loss did not improve from 0.76192

Epoch 51: val_loss did not improve from 0.76192
451/451 ━━━━━━━━━━━━━━━━━━━━ 797s 2


Epoch 63: finished saving model to CNN_Imagenes/Modelo/cnn_mejor.h5

Epoch 63: val_loss improved from 0.76192 to 0.76007, saving model to CNN_Imagenes/Modelo/cnn_pesos_mejor.weights.h5

Epoch 63: finished saving model to CNN_Imagenes/Modelo/cnn_pesos_mejor.weights.h5
451/451 ━━━━━━━━━━━━━━━━━━━━ 1170s 3s/step - accuracy: 0.7734 - loss: 0.6361 - mse: 0.0390 - val_accuracy: 0.7885 - val_loss: 0.7601 - val_mse: 0.0385
Epoch 64/100
451/451 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.7987 - loss: 0.5985 - mse: 0.0364
Epoch 64: val_loss did not improve from 0.76007

Epoch 64: val_loss did not improve from 0.76007
451/451 ━━━━━━━━━━━━━━━━━━━━ 1089s 2s/step - accuracy: 0.7912 - loss: 0.6126 - mse: 0.0377 - val_accuracy: 0.7769 - val_loss: 0.8718 - val_mse: 0.0401
Epoch 65/100
451/451 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.7971 - loss: 0.5704 - mse: 0.0353
Epoch 65: val_loss did not improve from 0.76007

Epoch 65: val_loss did not improve from 0.76007
451/451 ━━━━━━━━━━━━━━━━━━━━ 969s


Epoch 97: finished saving model to CNN_Imagenes/Modelo/cnn_mejor.h5

Epoch 97: val_loss improved from 0.76007 to 0.75367, saving model to CNN_Imagenes/Modelo/cnn_pesos_mejor.weights.h5

Epoch 97: finished saving model to CNN_Imagenes/Modelo/cnn_pesos_mejor.weights.h5
451/451 ━━━━━━━━━━━━━━━━━━━━ 797s 2s/step - accuracy: 0.8237 - loss: 0.5368 - mse: 0.0307 - val_accuracy: 0.8087 - val_loss: 0.7537 - val_mse: 0.0360
Epoch 98/100
451/451 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.8085 - loss: 0.5175 - mse: 0.0326
Epoch 98: val_loss did not improve from 0.75367

Epoch 98: val_loss did not improve from 0.75367
451/451 ━━━━━━━━━━━━━━━━━━━━ 829s 2s/step - accuracy: 0.8056 - loss: 0.5465 - mse: 0.0334 - val_accuracy: 0.7904 - val_loss: 1.0475 - val_mse: 0.0392
Epoch 99/100
451/451 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.8320 - loss: 0.5003 - mse: 0.0306
Epoch 99: val_loss did not improve from 0.75367

Epoch 99: val_loss did not improve from 0.75367
451/451 ━━━━━━━━━━━━━━━━━━━━ 1084s 

In [ ]:
#Guarda el modelo entrenado
CNN.save("CNN_Imagenes/Modelo/cnn.h5")
CNN.save_weights("CNN_Imagenes/Modelo/cnn_pesos.weights.h5")

# Nota.- En continuación a lo anterior, PERMITIMOS QUE,
# en caso de que los callbacks (ModelCheckpoint) no funcionen correctamente
# o el entrenamiento se interrumpa inesperadamente, se conserve una versión final
# del modelo entrenado junto con sus pesos. El guardado de aquí será lo oficial LO QUE SE USARA
#Los guardados previos sirven como respaldo únicamente

In [ ]:
#AQUÍ SE HARÁ LA EVALUACIÓN CON CAMARA (PRUEBA) - 
import numpy as np
from tensorflow.keras.utils import load_img, img_to_array
from keras.models import load_model
import os.path
import cv2

def evaluar(imagen):
    altura,anchura = 600,600
    modelo = "CNN_Imagenes/Modelo/cnn.h5"
    pesos = "CNN_Imagenes/Modelo/cnn_pesos.weights.h5"
    #Cargar la arquitectura de la cnn y sus pesos
    cnn = load_model(modelo)
    cnn.load_weights(pesos)
    #Clasificar la imagen o el objeto
    imagen_clasificar = cv2.resize(imagen,(anchura,altura))
    imagen_clasificar = imagen_clasificar/255
    imagen_clasificar = img_to_array(imagen_clasificar)
    imagen_clasificar = np.expand_dims(imagen_clasificar, axis=0)
    #Evaluar
    clase = cnn.predict(imagen_clasificar)
    arg_max = np.argmax(clase[0])
    # if (arg_max ==0):
    #     print("gallo")
    # elif(arg_max==1):
    #     print("perro")    

    # ===== ASEGURAR class_indices  =====
    class_indices = imagenes_entrenamiento.class_indices
    print(class_indices)

    # /*ESTO ES NUEVO - 20260410*/
    # ===== NORMALIZACIÓN  =====
    imagen_clasificar = imagen_clasificar / 255
    # ===== RE-EVALUAR CORRECTAMENTE  =====
    clase = cnn.predict(imagen_clasificar)
    arg_max = np.argmax(clase[0])
    # ===== OBTENER NOMBRES DE CLASES =====
    clases_lista = list(class_indices.keys())
    # ===== RESULTADO FINAL =====
    print("Clase predicha:", clases_lista[arg_max])
    print("Confianza:", round(clase[0][arg_max] * 100, 2), "%")


In [ ]:
#AQUÍ EVALUAREMOS CON CÁMARA (PRUEBA)
import cv2
capture = cv2.VideoCapture(0)
while(1):
    _,frame = capture.read() #leemos cada frame de la camara
    cv2.imshow("Ventana", frame)
    k = cv2.waitKey(5) & 0xFF #Esperamos una tecla
    if (k==27):
        break
    if (k==99):
        evaluar(frame) #Evaluamos la imagen capturada por la cámara al presionar la tecla 'k' (99 en código ASCII)
cv2.destroyAllWindows() 